# 三步质控快速版

奇偶深度校验 → 行星半径合理性 → NASA 档案交叉比对，**一步完成即时写 CSV**。

**提速手段**（相对分开运行的三个 notebook）：
- 奇偶校验：BLS 降分辨率 + 多扇区拼接 + 下载缓存
- 半径校验：TIC 星表**批量查询**（一次 `query_criteria` 替代 N 次 `query_object`）
- 交叉比对：缓存 + 名称/TIC ID 双键

**输出**：`vetting_result.csv`，列结构与之前 `nasa_crossmatch_result.csv` 完全一致。

## 0. 安装依赖

In [1]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与加载数据

In [2]:
import os, warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)

# ==================== 参数配置 ====================
ODD_EVEN_THRESHOLD   = 3.0          # σ > 3 -> 奇偶不通过
MAX_RP_GENERAL       = 2.5          # Rjup，通用行星半径上限
MAX_RP_MDWARF        = 2.0          # Rjup，M 矮星更严上限
R_SUN_TO_RJUP        = 9.7311       # 1 Rsun = 9.73 Rjup
R_JUP_TO_REARTH      = 11.2089      # 1 Rjup = 11.21 Rearth
STITCH_ALL_SECTORS   = True         # 拼接多扇区 -> 更多凌日
TESS_CACHE_DIR       = 'tess_cache'
OUTPUT_CSV           = 'outputs/output.csv'

os.makedirs(TESS_CACHE_DIR, exist_ok=True)

# ==================== 输入 ====================
_PATHS = ['inputs/put_files_here/input.csv',
          '02.csv',
          'inputs/ignore_these/01.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])

df = pd.read_csv(INPUT_CSV)
ORIG_DF = df.copy()                          # Person1 原始快照，末尾校验用

# Person1 的 8 个原始列
P1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
           'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

# 最终输出列顺序（与 nasa_crossmatch_result.csv 一致）
FINAL_COLS = P1_COLS + [
    'depth_odd_ppm', 'depth_even_ppm', 'n_transits',
    'odd_even_sigma', 'odd_even_reason', 'odd_even_pass',
    'TIC_ID', 'Stellar_Radius_Rsun', 'Teff_K', 'Spectral_Type',
    'Rp_Rjup', 'Rp_Rearth', 'radius_reason', 'planet_radius_plausible',
    '已知/新候选', 'match_source']

print(f'已加载 {INPUT_CSV}  ->  {len(df)} 个候选')
df.head()

已加载 inputs/put_files_here/input.csv  ->  99 个候选


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive
0,Kepler-1763,74,14.28905,0.51104,9085.6,0.50,4.47,True
1,TOI-7008,15,0.83972,0.00054,2396.3,1.10,8.39,False
2,Kepler-240,-1,7.93176,0.13773,726.0,2.95,7.73,False
3,Kepler-899,-1,19.14524,0.45475,616.5,6.00,6.58,True
4,TOI-4582,40,4.57931,0.12686,710.0,0.50,4.99,True


## 2. 校验一：奇偶深度检验（快速版）

In [3]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def _get_lc(target, sector):
    '''下载光变曲线(带缓存)，优先 SPOC + 多扇区拼接。'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    search = None
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'author': 'SPOC'},
               {'sector': int(sector)},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    if search is None:
        _LC_CACHE[key] = None
        return None
    lc = None
    try:
        if STITCH_ALL_SECTORS and len(search) > 1:
            lc = search.download_all(download_dir=TESS_CACHE_DIR).stitch()
        else:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        try:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
        except Exception:
            lc = None
    _LC_CACHE[key] = lc
    return lc

def odd_even_check(target, sector, period, duration_hr):
    '''返回 {depth_odd, depth_even, n_transits, sigma, reason}。'''
    out = {'depth_odd': np.nan, 'depth_even': np.nan, 'n_transits': 0,
           'sigma': np.nan, 'reason': ''}
    lc = _get_lc(target, sector)
    if lc is None:
        out['reason'] = 'no_lc'
        return out
    try:
        lc = lc.remove_nans().normalize().flatten(window_length=401)
        t = np.asarray(lc.time.value, dtype=float)
        f = np.asarray(lc.flux.value, dtype=float)
        g = np.isfinite(t) & np.isfinite(f)
        t, f = t[g], f[g]
        if len(t) < 200:
            out['reason'] = f'few_cad({len(t)})'
            return out
        mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
        scl = 1.4826 * mad if mad > 0 else np.nanstd(f)
        dy = np.full(len(t), scl if scl > 0 else 1e-3)
        bls = BoxLeastSquares(t, f, dy=dy)
        p0 = float(period)
        ps = np.linspace(p0 * 0.999, p0 * 1.001, 300)
        d0 = max(float(duration_hr), 0.5) / 24.0
        ds = np.unique(np.clip(np.array([d0 * 0.6, d0, d0 * 1.4]),
                               0.5 / 24.0, 8.0 / 24.0))
        res = bls.power(ps, ds, objective='snr')
        j = int(np.argmax(res.power))
        st = bls.compute_stats(float(res.period[j]),
                              float(res.duration[j]),
                              float(res.transit_time[j]))
        ntr = int(np.sum(np.atleast_1d(st['per_transit_count']) > 0))
        out['n_transits'] = ntr
        do, eo = float(st['depth_odd'][0]), float(st['depth_odd'][1])
        de, ee = float(st['depth_even'][0]), float(st['depth_even'][1])
        out['depth_odd'] = do * 1e6
        out['depth_even'] = de * 1e6
        comb = np.sqrt(eo ** 2 + ee ** 2)
        if not np.isfinite(comb) or comb <= 0:
            out['reason'] = f'err_undef(n={ntr})'
            return out
        out['sigma'] = abs(do - de) / comb
        out['reason'] = 'ok'
        return out
    except Exception as e:
        out['reason'] = f'ex:{type(e).__name__}'
        return out

In [4]:
print('=== 校验一：奇偶深度 ===')
ov = [(np.nan, np.nan, 0, np.nan, 'skipped')] * len(df)
for i, row in df.iterrows():
    r = odd_even_check(row['Target'], row['Sector'],
                       row['Period (d)'], row['Duration (hr)'])
    ov[i] = (r['depth_odd'], r['depth_even'], r['n_transits'], r['sigma'], r['reason'])
    tag = 'NaN' if not np.isfinite(r['sigma']) else f"{r['sigma']:.2f}"
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"sigma={tag:<6} n={r['n_transits']:<3} {r['reason']}")

d1, d2, ntr, sig, rsn = zip(*ov)
df['depth_odd_ppm'] = d1
df['depth_even_ppm'] = d2
df['n_transits'] = ntr
df['odd_even_sigma'] = sig
df['odd_even_reason'] = rsn
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_THRESHOLD) if np.isfinite(x) else pd.NA for x in sig],
    dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   通过: {int((df["odd_even_pass"]==True).sum())}  '
      f'不通过: {int((df["odd_even_pass"]==False).sum())}  '
      f'无法判定: {int(df["odd_even_pass"].isna().sum())}')

=== 校验一：奇偶深度 ===
[  1/99] Kepler-1763        sigma=0.59   n=2   ok
[  2/99] TOI-7008           sigma=0.70   n=29  ok
[  3/99] Kepler-240         sigma=2.00   n=185 ok
[  4/99] Kepler-899         sigma=2.66   n=76  ok
[  5/99] TOI-4582           sigma=0.52   n=6   ok
[  6/99] WASP-15            sigma=11.05  n=4   ok
[  7/99] TOI-6478           sigma=NaN    n=1   err_undef(n=1)
[  8/99] WASP-183           sigma=1.41   n=5   ok
[  9/99] KIC 9663113        sigma=0.73   n=2   ok
[ 10/99] K2-60              sigma=0.59   n=4   ok
[ 11/99] Kepler-220         sigma=1.02   n=6   ok
[ 12/99] Kepler-1887        sigma=0.15   n=6   ok
[ 13/99] TOI-5232           sigma=0.66   n=5   ok
[ 14/99] EPIC 206024342     sigma=2.92   n=2   ok
[ 15/99] Kepler-1844        sigma=0.30   n=13  ok
[ 16/99] Kepler-474         sigma=1.05   n=4   ok
[ 17/99] TOI-2158           sigma=4.07   n=3   ok
[ 18/99] K2-203             sigma=0.04   n=2   ok
[ 19/99] Kepler-648         sigma=2.16   n=5   ok
[ 20/99] TOI-238     

## 3. 校验二：行星半径合理性（批量 TIC 查询）

**提速关键**：识别目标名中的 TIC 编号，用 `Catalogs.query_criteria` 一次 SQL 查询拿回所有恒星参数，替代逐条 `query_object`。

In [5]:
from astroquery.mast import Catalogs

print('=== 校验二：行星半径 ===')

# ---- 辅助函数 ----
def _sf(x):
    try: return np.nan if x is None or np.ma.is_masked(x) else float(x)
    except: return np.nan

def _spectral(teff):
    if not np.isfinite(teff): return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T'

# ---- 分离两类目标：可批量查 TIC ID 的 vs 需按名查的 ----
STAR_RAD, STAR_TEFF, STAR_TIC = {}, {}, {}

tic_ids = []
name_targets = []
for _, row in df.iterrows():
    t = str(row['Target']).strip()
    if t.upper().startswith('TIC') and t[3:].strip().isdigit():
        tic_ids.append(int(t[3:].strip()))
    else:
        name_targets.append(t)

print(f'  TIC 编号目标(可批量): {len(tic_ids)}  名称目标(逐个): {len(name_targets)}')

# ---- 批量查 TIC ----
if tic_ids:
    ids_str = ','.join(str(x) for x in tic_ids)
    try:
        batch = Catalogs.query_criteria(
            catalog='TIC', select='ID,rad,Teff,mass,Tmag',
            where=f'ID IN ({ids_str})')
        if batch is not None:
            for r in batch:
                key = f'TIC {int(r["ID"])}'
                STAR_RAD[key] = _sf(r['rad'])
                STAR_TEFF[key] = _sf(r['Teff'])
                STAR_TIC[key] = int(r['ID'])
            print(f'  批量查询命中: {len(STAR_RAD)} 个')
    except Exception as e:
        print(f'  批量查询失败: {e}，将逐条查询...')
        for tid in tic_ids:
            try:
                cat = Catalogs.query_object(f'TIC {tid}', catalog='TIC', radius=0.02)
                if cat is not None and len(cat) > 0:
                    r_ = cat[0]
                    STAR_RAD[f'TIC {tid}'] = _sf(r_['rad'])
                    STAR_TEFF[f'TIC {tid}'] = _sf(r_['Teff'])
                    STAR_TIC[f'TIC {tid}'] = int(r_['ID'])
            except Exception:
                pass

# ---- 按名称逐个查 (少数) ----
for t in name_targets:
    try:
        cat = Catalogs.query_object(t, catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:
                order = np.argsort([_sf(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            STAR_RAD[t] = _sf(r['rad'])
            STAR_TEFF[t] = _sf(r['Teff'])
            STAR_TIC[t] = int(r['ID'])
    except Exception:
        pass

print(f'  总计查询到恒星参数: {len(STAR_RAD)} 个')

=== 校验二：行星半径 ===
  TIC 编号目标(可批量): 0  名称目标(逐个): 99
  总计查询到恒星参数: 99 个


In [6]:
# ---- 逐目标判定半径合理性 ----
rj_list, re_list, spec_list, rsn_list = [], [], [], []
tic_list, rstar_list, teff_list = [], [], []
plausible_list = []

for i, row in df.iterrows():
    t = str(row['Target']).strip()
    rad = STAR_RAD.get(t, np.nan)
    te = STAR_TEFF.get(t, np.nan)
    ti = STAR_TIC.get(t, None)
    sp = _spectral(te)
    d = float(row['Depth (ppm)']) * 1e-6

    if not np.isfinite(d) or d <= 0:
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'bad_depth'
    elif not np.isfinite(rad):
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'no_stellar_radius'
    else:
        rj = np.sqrt(d) * rad * R_SUN_TO_RJUP
        rre = rj * R_JUP_TO_REARTH
        mx = MAX_RP_MDWARF if sp == 'M' else MAX_RP_GENERAL
        if rj <= mx:
            ok, reason = True, 'plausible'
        else:
            ok, reason = False, ('too_large_mdwarf' if sp == 'M' else 'too_large')

    rj_list.append(rj); re_list.append(rre); spec_list.append(sp)
    rsn_list.append(reason); tic_list.append(ti)
    rstar_list.append(rad); teff_list.append(te)
    plausible_list.append(ok)

    rt = 'NaN' if not np.isfinite(rj) else f'{rj:.2f}'
    rs = 'NA' if not np.isfinite(rad) else f'{rad:.3f}'
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"R*={rs:<7} 型={sp:<5} Rp={rt:<7}Rjup  {str(ok):<5}")

df['TIC_ID'] = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K'] = teff_list
df['Spectral_Type'] = spec_list
df['Rp_Rjup'] = rj_list
df['Rp_Rearth'] = re_list
df['radius_reason'] = rsn_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   合理: {int((df["planet_radius_plausible"]==True).sum())}  '
      f'不合理: {int((df["planet_radius_plausible"]==False).sum())}  '
      f'无法判定: {int(df["planet_radius_plausible"].isna().sum())}')

[  1/99] Kepler-1763        R*=1.966   型=F     Rp=1.82   Rjup  True 
[  2/99] TOI-7008           R*=0.953   型=K     Rp=0.45   Rjup  True 
[  3/99] Kepler-240         R*=1.041   型=G     Rp=0.27   Rjup  True 
[  4/99] Kepler-899         R*=10.855  型=K     Rp=2.62   Rjup  False
[  5/99] TOI-4582           R*=2.569   型=G     Rp=0.67   Rjup  True 
[  6/99] WASP-15            R*=1.503   型=F     Rp=1.06   Rjup  True 
[  7/99] TOI-6478           R*=0.388   型=M     Rp=0.61   Rjup  True 
[  8/99] WASP-183           R*=0.916   型=G     Rp=0.49   Rjup  True 
[  9/99] KIC 9663113        R*=2.216   型=F     Rp=2.08   Rjup  True 
[ 10/99] K2-60              R*=11.014  型=K     Rp=5.11   Rjup  False
[ 11/99] Kepler-220         R*=10.736  型=K     Rp=4.88   Rjup  False
[ 12/99] Kepler-1887        R*=2.867   型=K     Rp=4.89   Rjup  False
[ 13/99] TOI-5232           R*=1.741   型=F     Rp=0.76   Rjup  True 
[ 14/99] EPIC 206024342     R*=0.848   型=G     Rp=0.75   Rjup  True 
[ 15/99] Kepler-1844        R*=2.2

## 4. 校验三：NASA 档案交叉比对

In [7]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

print('=== 校验三：NASA 档案交叉比对 ===')

_CACHE = {}

def _okt(x):
    try: return x is not None and not pd.isna(x) and int(x) > 0
    except: return False

def crossmatch(target, tic_id=None):
    '''返回 (已知/新候选, 命中来源)。'''
    key = (str(target), int(tic_id) if _okt(tic_id) else None)
    if key in _CACHE:
        return _CACHE[key]
    label, source = '新候选', 'none'
    # (1) 确认行星表：按名称
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label, source = '已知', 'confirmed'
    except Exception:
        source = 'query_error'
    # (2) TOI 表：按 TIC ID
    if label == '新候选' and _okt(tic_id):
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label, source = '已知', 'TOI'
        except Exception:
            if source != 'confirmed':
                source = 'query_error'
    _CACHE[key] = (label, source)
    return (label, source)

labels, sources = [], []
has_tic = 'TIC_ID' in df.columns
for i, row in df.iterrows():
    tic = row['TIC_ID'] if has_tic else None
    lb, so = crossmatch(row['Target'], tic)
    labels.append(lb); sources.append(so)
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} -> {lb:<6} ({so})")

df['已知/新候选'] = labels
df['match_source'] = sources

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   已知: {labels.count("已知")}  新候选: {labels.count("新候选")}')

=== 校验三：NASA 档案交叉比对 ===
[  1/99] Kepler-1763        -> 已知     (confirmed)
[  2/99] TOI-7008           -> 已知     (confirmed)
[  3/99] Kepler-240         -> 已知     (confirmed)
[  4/99] Kepler-899         -> 已知     (confirmed)
[  5/99] TOI-4582           -> 已知     (confirmed)
[  6/99] WASP-15            -> 已知     (confirmed)
[  7/99] TOI-6478           -> 已知     (confirmed)
[  8/99] WASP-183           -> 已知     (confirmed)
[  9/99] KIC 9663113        -> 已知     (confirmed)
[ 10/99] K2-60              -> 已知     (confirmed)
[ 11/99] Kepler-220         -> 已知     (confirmed)
[ 12/99] Kepler-1887        -> 已知     (confirmed)
[ 13/99] TOI-5232           -> 已知     (confirmed)
[ 14/99] EPIC 206024342     -> 已知     (confirmed)
[ 15/99] Kepler-1844        -> 已知     (confirmed)
[ 16/99] Kepler-474         -> 已知     (confirmed)
[ 17/99] TOI-2158           -> 已知     (confirmed)
[ 18/99] K2-203             -> 已知     (confirmed)
[ 19/99] Kepler-648         -> 已知     (confirmed)
[ 20/99] TOI-238          

## 5. 最终汇总与高置信度清单

In [8]:
# ---- Person1 数据完整性校验 ----
_p = [c for c in P1_COLS if c in df.columns]
assert df[_p].equals(ORIG_DF[_p]), 'Person1 原始数据被意外修改!'
print('✓ Person1 原始数据完整性校验通过\n')

# ---- 高置信度：奇偶通过 + 半径合理 + 非 Person1 原判假阳性 ----
fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
mask = (df['odd_even_pass'] == True) & (df['planet_radius_plausible'] == True) & (~fp)
hc = df[mask.values].copy()

# ---- 按 FINAL_COLS 列序写入最终结果 ----
final = df[[c for c in FINAL_COLS if c in df.columns]].copy()
final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
hc.to_csv('high_confidence_candidates.csv', index=False, encoding='utf-8-sig')

print('=' * 64)
print('三项质控汇总')
print('=' * 64)
print(f'候选总数     : {len(df)}')
print(f'奇偶通过     : {int((df["odd_even_pass"]==True).sum())}')
print(f'半径合理     : {int((df["planet_radius_plausible"]==True).sum())}')
print(f'已知/新候选   : {labels.count("已知")} / {labels.count("新候选")}')
print(f'>> 高置信度  : {len(hc)}  ->  high_confidence_candidates.csv')
print('=' * 64)
final.head(10)

✓ Person1 原始数据完整性校验通过

三项质控汇总
候选总数     : 99
奇偶通过     : 84
半径合理     : 72
已知/新候选   : 99 / 0
>> 高置信度  : 20  ->  high_confidence_candidates.csv


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible,已知/新候选,match_source
0,Kepler-1763,74,14.28905,0.51104,9085.6,0.50,4.47,True,8128.085558,10071.330726,2,0.592849,ok,True,138575130,1.965630,7007.00,F,1.823226,20.436358,plausible,True,已知,confirmed
1,TOI-7008,15,0.83972,0.00054,2396.3,1.10,8.39,False,2457.465736,2207.312138,29,0.697863,ok,True,335661164,0.952550,5170.00,K,0.453754,5.086081,plausible,True,已知,confirmed
2,Kepler-240,-1,7.93176,0.13773,726.0,2.95,7.73,False,106.511111,150.546725,185,2.002955,ok,True,122712430,1.040780,5409.00,G,0.272891,3.058808,plausible,True,已知,confirmed
3,Kepler-899,-1,19.14524,0.45475,616.5,6.00,6.58,True,215.708526,275.241226,76,2.656534,ok,True,169557708,10.854800,4440.58,K,2.622710,29.397696,too_large,False,已知,confirmed
4,TOI-4582,40,4.57931,0.12686,710.0,0.50,4.99,True,666.362512,812.787423,6,0.522841,ok,True,219854519,2.569350,5234.00,G,0.666215,7.467537,plausible,True,已知,confirmed
5,WASP-15,11,3.75238,0.02203,5282.2,3.55,83.17,False,5887.524236,4430.368229,4,11.046516,ok,False,111991770,1.503210,6169.00,F,1.063136,11.916590,plausible,True,已知,confirmed
6,TOI-6478,8,22.18168,0.73280,26056.5,2.95,24.22,False,0.000000,25170.171847,1,NaN,err_undef(n=1),<NA>,332657787,0.388431,3484.00,M,0.610146,6.839067,plausible,True,已知,confirmed
7,WASP-183,35,4.11054,0.03486,3032.4,1.10,8.73,False,2571.925859,3313.978693,5,1.406014,ok,True,14344979,0.916164,5424.82,G,0.490940,5.502897,plausible,True,已知,confirmed
8,KIC 9663113,74,17.93646,0.45669,9290.6,0.50,4.59,True,8457.828086,11333.515851,2,0.727615,ok,True,272844987,2.215620,6117.21,F,2.078161,23.293894,plausible,True,已知,confirmed
9,K2-60,42,3.00183,0.00944,2271.4,2.95,11.26,False,2414.472056,2150.645809,4,0.594413,ok,True,206284138,11.014400,4671.00,K,5.108220,57.257527,too_large,False,已知,confirmed
